In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer

In [2]:
#Dynamic search path configuration
import sys
import os

#Adjusting search path based on notebook's depth in directory --> iterates through directories until it finds directory name: US_conflict_forecasting
current_dir = os.getcwd()
while 'US_conflict_forecasting' not in os.path.basename(current_dir):
    current_dir = os.path.dirname(current_dir)

#Adjusting search path based on 'new' current directory
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

from project_setup import setup_project
project_root_dir = setup_project()

import pandas as pd

Current Project Root Directory set to: /Users/danie/Desktop/Hertie/2. Semester/1. ML/0. Final Project/US_conflict_forecasting
Current Search Path set to: ['c:\\Users\\danie\\Desktop\\Hertie\\2. Semester\\1. ML\\0. Final Project\\US_conflict_forecasting', 'c:\\Users\\danie\\Desktop\\Hertie\\2. Semester\\1. ML\\0. Final Project\\US_conflict_forecasting\\2_Dataset\\c_Merge_Cleaning', 'c:\\Users\\danie\\AppData\\Local\\Programs\\Python\\Python312\\python312.zip', 'c:\\Users\\danie\\AppData\\Local\\Programs\\Python\\Python312\\DLLs', 'c:\\Users\\danie\\AppData\\Local\\Programs\\Python\\Python312\\Lib', 'c:\\Users\\danie\\AppData\\Local\\Programs\\Python\\Python312', '', 'C:\\Users\\danie\\AppData\\Roaming\\Python\\Python312\\site-packages', 'C:\\Users\\danie\\AppData\\Roaming\\Python\\Python312\\site-packages\\win32', 'C:\\Users\\danie\\AppData\\Roaming\\Python\\Python312\\site-packages\\win32\\lib', 'C:\\Users\\danie\\AppData\\Roaming\\Python\\Python312\\site-packages\\Pythonwin', 'c:\\Use

In [3]:
# Load the data
#Adjust dataset name in import for dataset with lag (Final_Dataset_Lag.csv) or dataset without lag (Final_Dataset_noLag.csv)

data = pd.read_csv(os.path.join(project_root_dir, '2_Dataset/d_Final_Dataset/Final_Dataset_Lag.csv'))


In [4]:
# Drop some unnecessary columns
data = data.drop(columns = ['admin2', 'Post Office City Name'])

In [5]:
# Recode urban variable to categorical
data['Rural_Urban_Commerical Status'] = data['Rural_Urban_Commerical Status'].astype('category')

categories = ['Rural', 'Urban', 'Commerical']  # Specify your categories if you know them

# Initialize the LabelEncoder
label_encoder = LabelEncoder()

# Fit the LabelEncoder to your categories
label_encoder.fit(categories)

# Transform your variable to encoded values
data['Rural_Urban_Commerical Status'] = label_encoder.transform(data['Rural_Urban_Commerical Status'])

In [6]:
# Count missing values
nan_count = data.isna().sum().sum()
print("Number of NaN values:", nan_count)

Number of NaN values: 2088


In [8]:
# Find and count values that indicate '-'
def count_dash(series):
    return series.eq('-').sum()

count_dash_data = data.applymap(lambda x: x == '-').sum().sum()

print("Number of values with '-' in the entire DataFrame:", count_dash_data)

C:\Users\danie\AppData\Local\Temp\ipykernel_31084\1411631116.py:5: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  count_dash_data = data.applymap(lambda x: x == '-').sum().sum()


Number of values with '-' in the entire DataFrame: 12372


In [9]:
# Find columns that hold '-'
columns_with_dash = data.apply(lambda x: '-' in x.values)

# Save those columns to list
columns_with_dash = columns_with_dash[columns_with_dash].index.tolist()

# It's median / mean columns that are calculated from other variables

In [10]:
data['month'] = data['month'].apply(lambda x: int(str(x)[-2:]))

In [11]:
# Replace each '-' with NaN, so can be imputed later
data.replace('-', np.nan, inplace=True)

In [12]:
# Looking for columns with the string '250,000+'
columns_with_string = data.applymap(lambda x: isinstance(x, str) and '250,000+' in x).any()

# Save those columns to a list
columns_containing_string = columns_with_string[columns_with_string].index.tolist()

# Inspect
print("Columns containing '250,000+':", columns_containing_string)

C:\Users\danie\AppData\Local\Temp\ipykernel_31084\1338546616.py:2: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  columns_with_string = data.applymap(lambda x: isinstance(x, str) and '250,000+' in x).any()


Columns containing '250,000+': ['Median Household Income, 2022', 'Median Family Income, 2022']


In [13]:
# Replace '250,000+' with '250,000' in the specified columns
for column in columns_containing_string:
    data[column] = data[column].str.replace('250,000+', '250,000')

In [14]:
# Select any column that is non-numeric
non_numeric_columns = data.select_dtypes(exclude=['number']).columns.tolist()

# Transform the non-numeric columns to numeric

for column in non_numeric_columns:
    data[column] = pd.to_numeric(data[column], errors='coerce')

In [15]:
# Imputing missing values

imputer = SimpleImputer(missing_values=np.nan, strategy='mean')

# Fit the imputer on the data and transform the column in one step
data_imputed = imputer.fit_transform(data)


data_imputed = pd.DataFrame(data_imputed, columns=data.columns)

# Back to df
# data = pd.DataFrame(data)

#### Restructuring Sociodemographic Features to relative values

In [16]:
perc_of_population_cols = ['Males, 2022', 
                        'Females, 2022', 
                        '<18 Years Old, 2022', 
                        '18 to 34 Years Old, 2022', 
                        '35 to 64 Years Old, 2022', 
                        '65+ Years Old, 2022', 
                        'White, Not Hispanic or Latino, 2022', 
                        'Black or African American, Not Hispanic or Latino, 2022', 
                        'Asian, Not Hispanic or Latino, 2022', 
                        'Other Races, Not Hispanic or Latino, 2022', 
                        'Two or more races, 2022', 
                        'Hispanic or Latino (Any Race), 2022', 
                        'Native Born, 2022',
                        'Foreign Born, 2022', 
                        'Foreign Born, Naturalized U.S. Citizen, 2022', 
                        'Foreign Born, Not a U.S. Citizen', 
                        'K-12 Grade Students Enrolled in School, 2022', 
                        'K-12 Grade Students Enrolled in Public School, 2022', 
                        'K-12 Grade Students Enrolled in Private School, 2022', 
                        'No High School Diploma, 2022', 
                        'High School Diploma or GED, 2022', 
                        'Some College, No Degree, 2022', 
                        "Associate's Degree, 2022",
                        "Bachelor's Degree or Higher, 2022", 
                        'Veterans, 2022', 
                        'Nonveteran, 2022', 
                        'Veterans With a Service-Connected Disability Rating', 
                        'Total # People with Disability, 2022', 
                        '# Children (<18) With Disability, 2022', 
                        '# Working Age Adults (18-64) With Disability, 2022', 
                        '# Senior Citizens (65+) with Disability, 2022', 
                        'People <65 years Old with No Health Insurance', 
                        'Children (<19 Years Old) with No Health Insurance, 2022', 
                        'Young Adults (19-34 Years Old) with No Health Insurance, 2022', 
                        'Middle Age Adults (35 to 64 Years Old) with No Health Insurance Coverage, 2022', 
                        '# People in Labor Force, 2022', 
                        '# People Not in Labor Force, 2022', 
                        '# People in Civilian Labor Force, 2022', 
                        'People Employed', 
                        'People Unemployed', 
                        'Self-Employed, 2022', 
                        'Non-Self-Employed, 2022', 
                        'Employed in Agriculture, Forestry, Fishing and Hunting, 2022', 
                        'Employed in Mining, Quarrying, and Oil/Gas Extraction, 2022', 
                        'Employed in Construction, 2022', 
                        'Employed in Manufacturing, 2022', 
                        'Employed in Wholesale Trade, 2022', 
                        'Employed in Retail Trade', 
                        'Employed in Transportation and Warehousing, and Utilities, 2022', 
                        'Employed in Information (Publishing, Media, Software, etc.), 2022', 
                        'Employed in Finance, Insurance, and Real Estate/Leasing, 2022', 
                        'Employed in Professional, Scientific, Management, and Administrative etc., 2022', 
                        'Employed in Educational services, 2022', 
                        'Employed in  Health Care and Social Assistance, 2022', 
                        'Employed in Arts, Entertainment, and Recreation, 2022', 
                        'Employed in Accommodation and Food Services, 2022', 
                        'Employed in Other Services, 2022', 
                        'Employed in Public Administration, 2022', 
                        'People Below Poverty Level, 2022', 
                        'People At or Above Poverty Level, 2022', 
                        'Children (<18) Below Poverty Level, 2022', 
                        'Children (<18) At or Above Poverty Level, 2022', 
                        'Working Age Adults (18-64) Below Poverty Level, 2022', 
                        'Working Age Adults (18-64) At or Above Poverty Level, 2022', 
                        'Senior Citizens (65+) Below Poverty Level, 2022', 
                        'Senior Citizens (65+) At or Above Poverty Level, 2022', 
                        '<30% of Income for Rent', 
                        '30% to 49% of Income for Rent, 2022', 
                        '50%+ of Income for Rent, 2022']


In [17]:
#Converting columns as a share of population 
for column in perc_of_population_cols:
    data_imputed[column] = (data_imputed[column] / data_imputed['Total Population, 2022']) * 100
    data_imputed[column] = data_imputed[column].round(2)

In [18]:
perc_of_household_cols = ['Households with Internet Access, 2022',
                        'Households with No Internet Access, 2022',
                        'Married-couples with Own Children (<18) Households, 2022', 
                        'Married-couples with No Own Children (<18) Households, 2022', 
                        'Single Parents (No Spouse) with Children (<18) Households, 2022', 
                        'Single Persons Households (Living Alone),  2022',
                        'Other Types of Households, 2022', 
                        '% Households with Income from Wages and Salary, 2022',
                        '% Households with Income from Social Security income, 2022',
                        '% Households with Income from Supplemental Security Income (SSI), 2022',
                        '% Households with Income from Cash Public Assistance or SNAP, 2022',
                        '% Households with Income from Retirement Sources, 200',
                        'Household Income <$49,999',
                        'Households Income $50,000 to $99,999',
                        'Household Income $100,000 to $199,999',
                        'Household Income $200,000+',
                        'Homeowners  (Owner-Occupied Units), 2022',
                        'Renters (Renter-Occupied Units), 2022']

In [19]:
#Converting columns as a share of households 
for column in perc_of_household_cols:
    data_imputed[column] = (data_imputed[column] / data_imputed['# Households, 2022']) * 100
    data_imputed[column] = data_imputed[column].round(2)

In [20]:
#Dropping duplicate columns
data_imputed = data_imputed.drop(columns = ['Population, 2020',
                                            '# Square Land Miles, 2020',
                                            'Population per Square Land Mile, 2022',
                                            'Foreign Born, 2022.1',
                                            'Populating in Households, 2022',
                                            'Occupied Housing Units, 2022',
                                            '# Adults 25+ Years Old, 2022',
                                            '# People 18+ Years Old, 2022',
                                            'Nonveteran, 2022',
                                            'Veterans, 2022.1',
                                            'Veterans With No Service-Connected Disability Rating',
                                            '% Total # People with Disability, 2022',
                                            '% Children (<18) With Disability, 2022',
                                            '% Working Age Adults (18-64) With Disability, 2022',
                                            '% Senior Citizens (65+) with Disability, 2022',
                                            '# People <65 Years Old, 2022',
                                            'People <65 years Old with Health Insurance',
                                            '# Children (<19 Years Old), 2022',
                                            'Children (<19 Years Old) with Health Insurance, 2022',
                                            '# Young Adults (19-34 Years Old) , 2022',
                                            'Young Adults (19-34 Years Old) with Health Insurance, 2022',
                                            '# Middle Age Adults (35 to 64 Years Old), 2022',
                                            'Middle Age Adults (35 to 64 Years Old) with Health Insurance Coverage, 2022',
                                            'Total # Households',
                                            'Total # Families',
                                            'Total # Households.1',
                                            '# People 16+ Years Old, 2022',
                                            'Total # Employed People',
                                            'Total # Employed People, 2022',
                                            'Total # Employed People.1',
                                            'Worked from Home, 2022',
                                            'Did Not Work at Home, 2022',
                                            '# Employed Person Not Working at Home, 2022',
                                            'Drove Alone to Work, 2022',
                                            'Carpooled to Work, 2022',
                                            'Public Transportation to Work',
                                            'Walked or Bicycle to Work, 2022',
                                            'Taxicab, Motorcycle, or Other Means to Work, 2022',
                                            '# Employed People Not Worked at Home, 2022',
                                            'Average Commute Time to Work, 2022 (Minutes)',
                                            '# Households',
                                            'No vehicle available',
                                            '1 vehicle available',
                                            '2 vehicles available',
                                            '3+ vehicles available',
                                            'Total # People for Whom Poverty Status was Determined, 2022',
                                            '# Children (<18) For Whom Poverty Status Was Determined, 2022',
                                            'Working Age Adults (18-64) For Whom Poverty Status was Determined, 2022',
                                            'Senior Citizens (65+) For Whom Poverty Status Was Determined',
                                            '# Households, 2022.1',
                                            '# Households, 2022.2',
                                            'Total Housing Units, 2022',
                                            'Occupied Housing Units, 2022.1',
                                            'Vacant Housing Units, 2022',
                                            'Occupied Housing Units, 2022.2',
                                            'Total # Vacant Units, 2022',
                                            'Vacant Units for Sale or Rent, 2022',
                                            'Vacant Units for Seasonal, Rec., or Occasional Use, 2022',
                                            'Vacant Units or Other Reasons, 2022',
                                            #'# Housing Units',
                                            'Single Family Homes (1 Unit Detached), 2022',
                                            'Townhouse/Duplex, Row Homes (1 Unit Attached), 2022',
                                            'Small Apartment Type Building (2-5 Units), 2022',
                                            'Large Apartment Type Building (10+ Units), 2022',
                                            'Mobile Home & Other Types of Units (RV, Boat, etc.), 2022',
                                            '# Owner-Occupied Homeowners, 2022',
                                            '<$100,000 Owner-Occupied Homes Valued, 2022',
                                            '$100,000 to $199,999 Owner-Occupied Homes Valued, 2022',
                                            '$200,000 to $299,999 Owner-Occupied Homes Valued, 2022',
                                            '$300,000 to $499,999 Owner-Occupied Homes Valued, 2022',
                                            '$500,000+ Owner-Occupied Homes Valued',
                                            'Homeowners with Mortgage Whose Housing Costs was Calculated, 2022',
                                            'Homeowners with Mortgage Paying <30% Income for Housing Cost, 2022',
                                            'Homeowners with Mortgage Paying 30% to 49%  Income for Housing Cost, 2022',
                                            'Homeowners with Mortgage Paying 50%+  Income for Housing Cost, 2022',
                                            '# Renters for Whom Income to Rent Ratio was Determined, 2022', 
                                            '# Households Paying Cash Rent', 
                                            'ZCTA.1'])

In [21]:
#Creating cyclical time features by adding sine and cosine transformation of the month col

data_imputed['month'] = data_imputed['month'].astype(int)

data_imputed['month_sin'] = np.sin(2 * np.pi * data_imputed['month'] / 12)
data_imputed['month_cos'] = np.cos(2 * np.pi * data_imputed['month'] / 12)


In [22]:
nan_count = data_imputed.isna().sum().sum()
print("Number of NaN values:", nan_count)

Number of NaN values: 21984


In [23]:
# Impute again, as inctroduced 0s in relative columns create NaNs during the diviion operation
imputer = SimpleImputer(strategy='mean')  
data_imputed = pd.DataFrame(imputer.fit_transform(data_imputed), columns=data_imputed.columns)

In [24]:
# Save the final dataset to a CSV file
#Remeber to adjust filename based on whether you're exporting the lag or nolag dataset!
output_directory_1 = os.path.join(project_root_dir, '2_Dataset/d_Final_Dataset')
output_filename_1 = "Final_Dataset_Lag_clean.csv"
output_path_1 = os.path.join(output_directory_1, output_filename_1)

data_imputed.to_csv(output_path_1, index=False)